## Flow Cytometry Analysis — Combined 48h Experiments
### Author: Eleni Aretaki
### Date: 2026-03-24
### Purpose: 
    1. Load, preprocess, normalize, and merge two 48h flow cytometry datasets.
    2. Calculate KO/WT ratios, perform Levene's test, t-tests, and FDR-adjusted p-values.
    3. Generate phenotype-specific heatmaps annotated by MoA and significance.
#### Cell Lines: ARID1A, ARID1B, ARID2, SMARCA4, BRD9, WT (plus others partially screened)
#### Compounds: Camptothecin, MMS, Adavosertib, Palbociclib, MLN4924, Hydroxyurea, BIBR1523, Cobimetinib, Lapatinib, Niraparib, Paclitaxel

## PART 1 — Drug response heatmaps 

In [ ]:
# -------------------------------
# Imports
# -------------------------------
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import ttest_ind, levene
from statsmodels.stats.multitest import multipletests
import glob

In [ ]:
# ----------------------------
# Folder setup 
# ----------------------------
data_dir = '../data/'
out_dir_heatmaps = '../plots/Heatmaps/'
out_dir_barplots = '../plots/DMSO_Barplots/'
os.makedirs(out_dir_heatmaps, exist_ok=True)
os.makedirs(out_dir_barplots, exist_ok=True)

In [ ]:
# ----------------------------
# Read in log2-transformed datasets
# ----------------------------
fc1_log2 = pd.read_csv(os.path.join(data_dir, 'fc1_normalized_data.csv'), encoding='unicode_escape')
fc2_log2 = pd.read_csv(os.path.join(data_dir, 'fc2_normalized_data.csv'), encoding='unicode_escape')

fc1_log2["FC"] = 1
fc2_log2["FC"] = 2

# Remove treatments that were expired or inactive
fc1_log2 = fc1_log2[fc1_log2["treatment"] != "Aph"]
fc1_log2["treatment"] = fc1_log2["treatment"].str.replace("Pbc", "PBC")
fc2_log2 = fc2_log2[fc2_log2["treatment"] != "NIR"]

# Combine the two experiments
fc_complete = pd.concat([fc1_log2, fc2_log2], ignore_index=True)

# ----------------------------
# Define phenotypes
# ----------------------------
ratio_phenotypes = ['S-phase', 'G1-phase', 'G2-phase']
difference_phenotypes = ['Apoptosis-Positive', 'DSB-Positive']
all_phenotypes = ratio_phenotypes + difference_phenotypes

In [ ]:
# -------------------------------
# Statistical Functions
# -------------------------------

def calculate_levene(df, phenotypes):
    """Compute Levene's test for KO vs WT for each treatment and phenotype."""
    results = []
    for (fc, treatment), group in df.groupby(['FC', 'treatment']):
        if treatment == 'DMSO':
            continue
        mods = group['modification'].unique()
        for mod in mods:
            if mod == 'C631':
                continue
            ko_data = group[group['modification'] == mod]
            wt_data = group[group['modification'] == 'C631']
            for pheno in phenotypes:
                if len(ko_data) > 1 and len(wt_data) > 1:
                    stat, p = levene(ko_data[pheno], wt_data[pheno], center='median')
                    eq_var = p >= 0.05
                else:
                    p = None
                    eq_var = None
                results.append({
                    'FC': fc,
                    'Treatment': treatment,
                    'Cell_Line': mod,
                    'Phenotype': pheno,
                    'Levene_P': p,
                    'Equal_Var': eq_var
                })
    return pd.DataFrame(results)

lev_res = calculate_levene(fc_complete, all_phenotypes)

def calculate_p_values(df, phenotypes):
    """Compute t-tests for KO vs WT."""
    results = []
    for (fc, treatment), group in df.groupby(['FC', 'treatment']):
        if treatment == 'DMSO':
            continue
        mods = group['modification'].unique()
        for mod in mods:
            if mod == 'C631':
                continue
            ko_data = group[group['modification'] == mod]
            wt_data = group[group['modification'] == 'C631']
            for pheno in phenotypes:
                if len(ko_data) > 1 and len(wt_data) > 1:
                    t_stat, p_val = ttest_ind(ko_data[pheno], wt_data[pheno])
                else:
                    t_stat, p_val = None, None
                results.append({
                    'FC': fc,
                    'Treatment': treatment,
                    'Cell_Line': mod,
                    'Phenotype': pheno,
                    'T-Statistic': t_stat,
                    'P-Value': p_val
                })
    return pd.DataFrame(results)

pvals_df = calculate_p_values(fc_complete, all_phenotypes)

# Map phenotypes to groups
phenotype_groups = {'Cell Cycle': ratio_phenotypes, 'DNA Damage': difference_phenotypes}
pvals_df['Phenotype Group'] = pvals_df['Phenotype'].map({pheno: group for group, phenos in phenotype_groups.items() for pheno in phenos})

# Adjust p-values

def adjust_p_values(results_df, method='fdr_bh'):
    valid_mask = results_df['P-Value'].notnull()
    pvals = results_df.loc[valid_mask, 'P-Value']
    _, adj, _, _ = multipletests(pvals, method=method)
    results_df.loc[valid_mask, 'Adjusted_P'] = adj
    return results_df

pvals_adj = pvals_df.groupby('Phenotype Group').apply(lambda g: adjust_p_values(g)).reset_index(drop=True)

In [ ]:
# ----------------------------
# Compute KO/WT ratios
# ----------------------------
def calculate_ko_wt_log2_ratios(df, phenotypes):
    """Compute KO/WT log2 ratios per treatment, replicate, DMSO concentration."""
    results = []
    for (fc, replicate, dmso_conc), group in df.groupby(['FC', 'replicate', 'DMSO_conc']):
        group = group[(group['treatment'] != 'DMSO') | (group['modification'] != 'C631')]
        for treatment in group['treatment'].unique():
            ko_data = group[(group['modification'] != 'C631') & (group['treatment'] == treatment)]
            wt_data = group[(group['modification'] == 'C631') & (group['treatment'] == treatment)]
            if not ko_data.empty and not wt_data.empty:
                for cl in ko_data['modification'].unique():
                    ko_vals = ko_data[ko_data['modification'] == cl]
                    wt_vals = wt_data
                    ratio_row = {
                        'FC': fc, 'Experiment': replicate, 'DMSO_conc': dmso_conc,
                        'Treatment': treatment, 'Cell_Line': cl
                    }
                    for pheno in phenotypes:
                        ko_val = ko_vals[pheno].values[0]
                        wt_val = wt_vals[pheno].values[0]
                        ratio_row[pheno] = ko_val - wt_val  # log2(KO/WT)
                    results.append(ratio_row)
    return pd.DataFrame(results)

log2_ratios_df = calculate_ko_wt_log2_ratios(fc_complete, all_phenotypes)

# -------------------------------
# Merge Ratios with Adjusted P-Values
# -------------------------------
final_ratios_long = log2_ratios_df.melt(
    id_vars=['FC', 'Experiment', 'DMSO_conc', 'Treatment', 'Cell_Line'],
    var_name='Phenotype', value_name='Log2_KO_WT_Ratio'
)
final_ratios_long['Phenotype'] = final_ratios_long['Phenotype'].str.replace('_KO_WT_ratio', '', regex=False)

final_results = pd.merge(final_ratios_long, pvals_adj, 
                         on=['FC','Treatment', 'Cell_Line', 'Phenotype'], 
                         how='inner')

# Average across replicates
numeric_columns = final_results.select_dtypes(include=['number']).columns
final_results_avg = final_results.groupby(['FC','Treatment', 'Cell_Line', 'Phenotype'], as_index=False)[numeric_columns].mean()

In [ ]:
# -------------------------------
# MoA & Heatmap Preparation
# -------------------------------
fc = final_results_avg.copy()
fc['Treatment'] = fc['Treatment'].str.replace('PAC', 'PTX').str.replace('Ada', 'ADA')

moa_dict = {"ADA": "G2/M phase", "BIBR": "Telomerase inhibitor", "CPT": "TopI inhibitor", "HU": "S phase",
            "MLN": "Protein homeostasis", "MMS": "DNA alkylating", "PBC": "G1/S phase",
            "COB": "MEK inhibitor", "LAP": "EGFR inhibitor", "NIR": "PARP inhibitor", "PTX": "G2/M phase"}
moa_to_color = {"G2/M phase": "#94C11F", "Telomerase inhibitor": "#030405", "TopI inhibitor": "#A55199",
                "S phase": "#2AA437", "Protein homeostasis": "#BE1522", "DNA alkylating": "#F7BDCA",
                "G1/S phase": "#BAD8B2", "MEK inhibitor": "#2DAAE1", "EGFR inhibitor": "#0F70B7",
                "PARP inhibitor": "#E1770B"}

fc['MoA'] = fc['Treatment'].map(moa_dict)
fc['MoA_color'] = fc['MoA'].map(moa_to_color)

fc['Significance'] = fc['Adjusted_P'].apply(lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')

heatmap_data_dict = {pheno: fc[fc['Phenotype'] == pheno].pivot(index='Treatment', columns='Cell_Line', values='Log2_KO_WT_Ratio') for pheno in fc['Phenotype'].unique()}
significance_dict = {pheno: fc[fc['Phenotype'] == pheno].pivot(index='Treatment', columns='Cell_Line', values='Significance') for pheno in fc['Phenotype'].unique()}

In [ ]:
# -------------------------------
# Treatment ordering by MoA
# -------------------------------

moa_order = [
    "G1/S phase",
    "S phase",
    "G2/M phase",
    "EGFR inhibitor",
    "MEK inhibitor",
    "DNA alkylating",
    "TopI inhibitor",
    "Protein homeostasis",
    "PARP inhibitor",
    "Telomerase inhibitor"
]

fc["MoA"] = pd.Categorical(fc["MoA"], categories=moa_order, ordered=True)

treatment_moa_table = (
    fc[["Treatment", "MoA", "MoA_color"]]
    .drop_duplicates()
)

treatment_moa_table["MoA_sort"] = (
    treatment_moa_table["MoA"].cat.codes.replace(-1, len(moa_order))
)

ordered_treatments = (
    treatment_moa_table
    .sort_values(["MoA_sort", "Treatment"])
    ["Treatment"]
    .tolist()
)

In [ ]:
# -------------------------------
# Heatmap Plot Function
# -------------------------------

def plot_phenotype_heatmap(phenotype, heatmap_data_dict, significance_dict, ordered_treatments, ordered_cell_lines, out_dir):
    plt.rcParams['font.family'] = 'Arial'
    
    heatmap_data = heatmap_data_dict[phenotype].loc[ordered_treatments, ordered_cell_lines]
    significance_data = significance_dict[phenotype].loc[ordered_treatments, ordered_cell_lines]

    # Replace KO suffixes for clarity
    heatmap_data.columns = [col.replace(' KO','') for col in heatmap_data.columns]
    significance_data.columns = [col.replace(' KO','') for col in significance_data.columns]

    # Determine color scaling
    if phenotype in ratio_phenotypes:
        norm = mcolors.TwoSlopeNorm(vmin=-2, vcenter=0, vmax=2)
        cbar_label = 'log2(KO/WT)'
    else:
        norm = mcolors.TwoSlopeNorm(vmin=-20, vcenter=0, vmax=20)
        cbar_label = 'KO - WT'

    # ----------------------------
    # Plotting
    # ----------------------------
    fig, ax = plt.subplots(figsize=(2.8, 3.2))
    sns.heatmap(heatmap_data, cmap='coolwarm', annot=significance_data, fmt='', annot_kws={'fontsize':8}, linewidths=0.5, norm=norm, ax=ax, cbar_kws={'label': cbar_label, 'shrink':0.4})

    # MoA annotation bar
    row_colors = (
        treatment_moa_table
        .set_index("Treatment")
        .loc[heatmap_data.index, "MoA_color"]
    )

    for y, color in enumerate(row_colors):
        ax.add_patch(
            plt.Rectangle(
                (-0.3, y),
                0.3,
                1,
                facecolor=color,
                edgecolor="none",
                clip_on=False
            )
        )
        
    # Style and labels
    colorbar = ax.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=7)
    colorbar.set_label(cbar_label, fontsize=8)
    
    ax.set_title(phenotype, fontsize=9)
    ax.tick_params(axis='y', labelsize=7, width=0.4)
    ax.set_xticklabels([t.get_text().replace("C631 + BRM014", "WT+BRM014") for t in ax.get_xticklabels()], rotation=90, fontsize=7)
    ax.tick_params(axis='x', labelsize=7, width=0.4)
    ax.set_xlabel("Knockout", fontsize=8)
    ax.set_ylabel("Treatment", fontsize=8)
    ax.tick_params(axis="y", pad=10)
    
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{phenotype}_heatmap.pdf'), dpi=600)
    plt.close(fig)

In [ ]:
# -------------------------------
# Generate Heatmaps
# -------------------------------

# Example ordering (customizable)
ordered_cell_lines = ['ARID1A KO', 'ARID1B KO', 'ARID2 KO', 'PBRM1 KO', 'BRD9 KO', 'SMARCC1 KO', 'SMARCA4 KO']

for pheno in fc['Phenotype'].unique():
    plot_phenotype_heatmap(pheno, heatmap_data_dict, significance_dict, ordered_treatments, ordered_cell_lines, out_dir_heatmaps)

## PART 2 - Baseline DMSO comparison

In [ ]:
# ----------------------------
# Read in raw-percentage datasets
# ----------------------------
fc1_raw = pd.read_csv(os.path.join(data_dir, 'fc1_combined_data.csv'), encoding='unicode_escape')
fc2_raw = pd.read_csv(os.path.join(data_dir, 'fc2_combined_data.csv'), encoding='unicode_escape')

fc1_raw["FC"] = 1
fc2_raw["FC"] = 2

# Remove treatments that were expired or inactive
fc1_raw = fc1_raw[fc1_raw["treatment"] != "Aph"]
fc1_raw["treatment"] = fc1_raw["treatment"].str.replace("Pbc", "PBC")
fc2_raw = fc2_raw[fc2_raw["treatment"] != "NIR"]

# Combine the two experiments
fc_raw_complete = pd.concat([fc1_raw, fc2_raw], ignore_index=True)

color_mapping = {
    "C631": "#9C9B9B", 
    "ARID1A KO": "#C52030",
    "ARID1B KO": "#8A181A",
    "ARID2 KO": "#176533",
    "BRD9 KO": "#D2AE2A",
    "SMARCA4 KO": "#80519B",
    "SMARCC1 KO": "#3569A1",
    "SMARCD1 KO": "#9ABCDC",
    "PBRM1 KO": "#93C13E",
    "BRD7 KO": "#566B30", 
    "PHF10 KO": "#4C491D",
    'C631 + BRM014': 'gray'
}

custom_order = [
    "C631", "ARID1A KO", "ARID1B KO", "ARID2 KO", "BRD7 KO", "PHF10 KO", "PBRM1 KO",
    "BRD9 KO", "SMARCD1 KO", "SMARCC1 KO", "SMARCA4 KO", 'C631 + BRM014'
]

In [ ]:
# ----------------------------
# Filter DMSO only
# ----------------------------
dmso_data = fc_raw_complete[
    fc_raw_complete["treatment"] == "DMSO"
].copy()

# ----------------------------
# Summary statistics
# ----------------------------
summary = (
    dmso_data
    .groupby("modification")[all_phenotypes]
    .agg(["mean","sem"])
    .reset_index()
)

summary.columns = [
    "_".join(col).strip("_")
    for col in summary.columns
]

summary.rename(
    columns={"modification_":"modification"},
    inplace=True
)

In [ ]:
# ----------------------------
# Order cell lines (same as heatmaps)
# ----------------------------
available_cell_lines = [
    c for c in custom_order
    if c in summary["modification"].values
]

summary["modification"] = pd.Categorical(
    summary["modification"],
    categories=available_cell_lines,
    ordered=True
)

summary = (
    summary
    .set_index("modification")
    .loc[available_cell_lines]
    .reset_index()
)

In [ ]:
# ----------------------------
# Statistics
# ----------------------------
p_values = []
pairs = []

# T-test KO vs WT
for phenotype in all_phenotypes:

    wt_data = dmso_data[
        dmso_data["modification"] == "C631"
    ][phenotype]

    for ko in dmso_data["modification"].unique():

        if ko == "C631":
            continue

        ko_data = dmso_data[
            dmso_data["modification"] == ko
        ][phenotype]

        if ko_data.empty:
            continue

        t_stat, p_val = ttest_ind(
            wt_data,
            ko_data,
            nan_policy="omit"
        )

        p_values.append(p_val)
        pairs.append((phenotype, ko))

# FDR correction (same logic as heatmaps)
adjusted_p_values = {}

for group_name, phenos in phenotype_groups.items():

    group_indices = [
        i for i,(phen,_) in enumerate(pairs)
        if phen in phenos
    ]

    group_pvals = [p_values[i] for i in group_indices]

    _, adj, _, _ = multipletests(
        group_pvals,
        method="fdr_bh"
    )

    for idx, adj_p in zip(group_indices, adj):
        adjusted_p_values[idx] = adj_p

adjusted_p_values = [
    adjusted_p_values.get(i,None)
    for i in range(len(pairs))
]

In [ ]:
# ----------------------------
# Plotting
# ----------------------------
plt.rcParams['font.family'] = 'Arial'

# Significance marker helper
def significance_marker(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''
        
def plot_dmso_barplot(phenotype):

    means = summary[f"{phenotype}_mean"]
    errors = summary[f"{phenotype}_sem"]

    fig, ax = plt.subplots(figsize=(3.5,3.25))

    bars = ax.bar(
        summary["modification"],
        means,
        yerr=errors,
        capsize=1.2,
        color=[
            color_mapping.get(x,"gray")
            for x in summary["modification"]
        ],
        error_kw={
            "elinewidth":0.4,
            "capthick":0.4
        }
    )

    # WT reference
    if "C631" in summary["modification"].values:
        wt_idx = summary["modification"].tolist().index("C631")
        ax.axhline(
            means[wt_idx],
            linestyle="--",
            linewidth=0.5,
            color="gray"
        )

    # significance markers
    for j, label in enumerate(summary["modification"]):

        for idx,(phen,ko) in enumerate(pairs):

            if phen == phenotype and ko == label:

                p = adjusted_p_values[idx]
                marker = significance_marker(p)

                if marker:
                    ax.text(
                        j,
                        means[j] + errors[j],
                        marker,
                        ha="center",
                        fontsize=10
                    )

    ax.set_ylabel(f"{phenotype} cells (%)", fontsize=8)
    ax.tick_params(axis='y', labelsize=7, width=0.4)
    ax.set_xlabel("Knockout", fontsize=8)

    ax.set_xticklabels(
        summary["modification"]
        .str.replace("C631","WT")
        .str.replace(" KO","")
        .str.replace("C631 + BRM014","WT+BRM014"),
        rotation=90, fontsize=7
    )
    ax.tick_params(axis='x', labelsize=7, width=0.4)
    
     # Spines
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            out_dir_barplots,
            f"{phenotype}_DMSO_barplot.pdf"
        ),
        dpi=600
    )

    plt.close()

for phenotype in all_phenotypes:
    plot_dmso_barplot(phenotype)